# 第四题：实现 SPODE 分类器

对应实验 7.4。在 Iris 数据集上，以四个属性分别作为超父，比较分层十折交叉验证的
accuracy、查准率、查全率和 F1，并选出本次实验中最好的超父。

## 连续特征的高斯 SPODE 模型

SPODE（Super-Parent One-Dependence Estimator）允许其他属性依赖同一个超父属性 $x_s$：

$$p(y,x)=P(y)\,p(x_s\mid y)\prod_{j\ne s}p(x_j\mid y,x_s).$$

与高斯朴素贝叶斯相比，它保留了超父与每个子属性在给定类别下的相关性。
Iris 为连续变量，本实验采用高斯条件密度，不把连续值直接当成离散类别计数。
假设在每个类别内，超父和每个子属性服从二元高斯分布，则

$$x_s\mid y=c\sim\mathcal N(\mu_{c,s},v_{c,s}),$$
$$x_j\mid x_s,y=c\sim\mathcal N\left(
\mu_{c,j}+\frac{\operatorname{cov}_c(x_j,x_s)}{v_{c,s}}(x_s-\mu_{c,s}),
v_{c,j}-\frac{\operatorname{cov}_c(x_j,x_s)^2}{v_{c,s}}\right).$$

各个均值、方差和协方差只在该折训练集内估计；对方差加微小正数以防数值问题。
预测时将先验、超父密度和各子属性条件密度取对数后相加，选取得分最大的类别。


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

data_path = next(
    (base / "data" / "iris" / "iris.data"
     for base in (Path.cwd(), Path.cwd().parent)
     if (base / "data" / "iris" / "iris.data").is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("请从项目根目录或 solution 目录运行本 Notebook。")
# Iris 最后一列是字符串类别，不能把整张表直接读成浮点数。
iris = np.loadtxt(data_path, delimiter=",", dtype=str)
irisX = iris[:, :4].astype(float)
irisY = iris[:, 4]
feature_names = ["萼片长度", "萼片宽度", "花瓣长度", "花瓣宽度"]
assert irisX.shape == (150, 4)
assert len(np.unique(irisY)) == 3 and np.isfinite(irisX).all()
print(f"Iris: {irisX.shape[0]} 个样本，{irisX.shape[1]} 个特征")
print(dict(zip(*np.unique(irisY, return_counts=True))))


Iris: 150 个样本，4 个特征
{'Iris-setosa': 50, 'Iris-versicolor': 50, 'Iris-virginica': 50}


In [2]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from scipy.special import logsumexp

class GaussianSPODE(ClassifierMixin, BaseEstimator):
    """超父为指定连续属性的高斯 SPODE，参数估计和预测均手写实现。"""

    def __init__(self, super_parent=0, var_smoothing=1e-9):
        self.super_parent = super_parent
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X, y = check_X_y(X, y, dtype=float)
        if not isinstance(self.super_parent, (int, np.integer)):
            raise ValueError("超父必须为整数列号。")
        if not 0 <= self.super_parent < X.shape[1]:
            raise ValueError("超父列号超出特征范围。")
        if not np.isfinite(self.var_smoothing) or self.var_smoothing <= 0:
            raise ValueError("var_smoothing 必须为有限正数。")
        self.n_features_in_ = X.shape[1]
        self.classes_, indices = np.unique(y, return_inverse=True)
        self.class_prior_ = np.bincount(indices) / len(y)
        self.children_ = np.array([
            j for j in range(X.shape[1]) if j != self.super_parent
        ], dtype=int)
        self.epsilon_ = max(
            self.var_smoothing * np.var(X, axis=0).max(), np.finfo(float).eps
        )
        means, parent_vars, coefficients, residual_vars = [], [], [], []
        for k in range(len(self.classes_)):
            subset = X[indices == k]
            mean = subset.mean(axis=0)
            centered = subset - mean
            # 最大似然协方差，分母为当前类别训练样本数。
            covariance = centered.T @ centered / len(subset)
            parent_var = covariance[self.super_parent, self.super_parent] + self.epsilon_
            parent_cov = covariance[self.children_, self.super_parent]
            coefficient = parent_cov / parent_var
            residual_var = (
                np.diag(covariance)[self.children_] - parent_cov ** 2 / parent_var
            )
            # 浮点舍入或完全相关时，条件方差也需保持严格为正。
            residual_var = np.maximum(residual_var, self.epsilon_)
            means.append(mean)
            parent_vars.append(parent_var)
            coefficients.append(coefficient)
            residual_vars.append(residual_var)
        self.means_ = np.asarray(means)
        self.parent_vars_ = np.asarray(parent_vars)
        self.coefficients_ = np.asarray(coefficients)
        self.residual_vars_ = np.asarray(residual_vars)
        return self

    def _joint_log_likelihood(self, X):
        check_is_fitted(self, "means_")
        X = check_array(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("训练和预测的特征数不一致。")
        scores = np.empty((len(X), len(self.classes_)))
        for k in range(len(self.classes_)):
            parent_delta = X[:, self.super_parent] - self.means_[k, self.super_parent]
            # 超父属性在当前类别下的高斯对数密度。
            parent_log = -0.5 * (
                np.log(2 * np.pi * self.parent_vars_[k])
                + parent_delta ** 2 / self.parent_vars_[k]
            )
            conditional_mean = (
                self.means_[k, self.children_]
                + parent_delta[:, None] * self.coefficients_[k]
            )
            # 给定类别和超父后，子属性条件独立，故对数密度可以求和。
            child_log = -0.5 * np.sum(
                np.log(2 * np.pi * self.residual_vars_[k])
                + (X[:, self.children_] - conditional_mean) ** 2 / self.residual_vars_[k],
                axis=1,
            )
            scores[:, k] = np.log(self.class_prior_[k]) + parent_log + child_log
        return scores

    def predict_proba(self, X):
        scores = self._joint_log_likelihood(X)
        return np.exp(scores - logsumexp(scores, axis=1, keepdims=True))

    def predict(self, X):
        scores = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(scores, axis=1)]


## 比较四种超父选择

Iris 是三分类问题。查准率、查全率和 F1 使用 **macro 平均**：先分别计算各类别的指标，
再等权平均。四种模型共用 `shuffle=True, random_state=42` 的分层十折划分。
主表报告各折指标的均值及样本标准差，选择依据依次是平均 accuracy、平均 macro F1、较小列号。


In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(cv.split(irisX, irisY))
records = []
oof_predictions = {}
metric_names = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
for parent, parent_name in enumerate(feature_names):
    predicted_all = np.empty_like(irisY)
    for fold, (train_index, valid_index) in enumerate(splits, 1):
        model = GaussianSPODE(super_parent=parent).fit(
            irisX[train_index], irisY[train_index]
        )
        predicted = model.predict(irisX[valid_index])
        truth = irisY[valid_index]
        predicted_all[valid_index] = predicted
        records.append({
            "parent": parent, "parent_name": parent_name, "fold": fold,
            "accuracy": accuracy_score(truth, predicted),
            "precision_macro": precision_score(truth, predicted, average="macro", zero_division=0),
            "recall_macro": recall_score(truth, predicted, average="macro", zero_division=0),
            "f1_macro": f1_score(truth, predicted, average="macro", zero_division=0),
        })
        # 概率归一化验证也在各折独立测试数据上进行。
        np.testing.assert_allclose(model.predict_proba(irisX[valid_index]).sum(axis=1), 1.0)
    oof_predictions[parent] = predicted_all

fold_results = pd.DataFrame(records)
cv_mean = fold_results.groupby(["parent", "parent_name"])[metric_names].mean()
cv_std = fold_results.groupby(["parent", "parent_name"])[metric_names].std(ddof=1)
print("分层十折指标均值")
display(cv_mean.round(6))
print("各折指标的样本标准差")
display(cv_std.round(6))
ranked = cv_mean.reset_index().sort_values(
    ["accuracy", "f1_macro", "parent"], ascending=[False, False, True]
)
best_parent = int(ranked.iloc[0]["parent"])
print(f"本次十折比较选择的超父：第 {best_parent + 1} 个属性（{feature_names[best_parent]}）")
labels = np.unique(irisY)
print("所选超父的折外预测混淆矩阵；行是真实类别，列是预测类别")
display(pd.DataFrame(
    confusion_matrix(irisY, oof_predictions[best_parent], labels=labels),
    index=labels, columns=labels,
))
assert len(records) == 40


分层十折指标均值


,,accuracy,precision_macro,recall_macro,f1_macro
parent,parent_name,,,,
0,萼片长度,0.966667,0.970000,0.966667,0.966465
1,萼片宽度,0.966667,0.973810,0.966667,0.965909
2,花瓣长度,0.980000,0.981111,0.980000,0.979933
3,花瓣宽度,0.973333,0.975556,0.973333,0.973199


各折指标的样本标准差


,,accuracy,precision_macro,recall_macro,f1_macro
parent,parent_name,,,,
0,萼片长度,0.047140,0.044767,0.047140,0.047300
1,萼片宽度,0.047140,0.035700,0.047140,0.048617
2,花瓣长度,0.044997,0.043838,0.044997,0.045075
3,花瓣宽度,0.046614,0.044691,0.046614,0.046743


本次十折比较选择的超父：第 3 个属性（花瓣长度）
所选超父的折外预测混淆矩阵；行是真实类别，列是预测类别


,Iris-setosa,Iris-versicolor,Iris-virginica
Iris-setosa,50,0,0
Iris-versicolor,0,48,2
Iris-virginica,0,1,49


In [4]:
# 独立数学核对：SPODE 的一类联合密度可写成具有星形依赖结构的多元高斯。
# 与 scipy 的多元高斯 logpdf 比较，检查条件均值、条件方差和求和公式。
from scipy.stats import multivariate_normal

check_train, check_valid = splits[0]
check_model = GaussianSPODE(super_parent=best_parent).fit(
    irisX[check_train], irisY[check_train]
)
checkX = irisX[check_valid]
joint = check_model._joint_log_likelihood(checkX)
for k in range(len(check_model.classes_)):
    s = check_model.super_parent
    children = check_model.children_
    v = check_model.parent_vars_[k]
    beta = check_model.coefficients_[k]
    covariance = np.zeros((irisX.shape[1], irisX.shape[1]))
    covariance[s, s] = v
    covariance[children, s] = covariance[s, children] = beta * v
    covariance[np.ix_(children, children)] = (
        np.diag(check_model.residual_vars_[k]) + v * np.outer(beta, beta)
    )
    expected = (
        np.log(check_model.class_prior_[k])
        + multivariate_normal.logpdf(checkX, mean=check_model.means_[k], cov=covariance)
    )
    np.testing.assert_allclose(joint[:, k], expected, rtol=1e-9, atol=1e-9)
print("验证通过：条件密度乘积与对应多元高斯的对数密度一致。")


验证通过：条件密度乘积与对应多元高斯的对数密度一致。


## 结果与分析

| 属性编号（从 1 开始） | 超父属性 | accuracy | precision macro | recall macro | F1 macro |
| --- | --- | --- | --- | --- | --- |
| 1 | 萼片长度 | 0.966667 | 0.970000 | 0.966667 | 0.966465 |
| 2 | 萼片宽度 | 0.966667 | 0.973810 | 0.966667 | 0.965909 |
| 3 | 花瓣长度 | 0.980000 | 0.981111 | 0.980000 | 0.979933 |
| 4 | 花瓣宽度 | 0.973333 | 0.975556 | 0.973333 | 0.973199 |

本次比较选出第 3 个属性 **花瓣长度** 作为超父：十折平均 accuracy 为 0.980000，宏平均查准率为 0.981111，宏平均查全率为 0.980000，宏平均 F1 为 0.979933。选择使用预先规定的排序规则，不根据全数据拟合准确率选择超父。高斯条件密度乘积已与相应星形协方差多元高斯的对数密度独立核对一致。

不同超父使用不同的依赖结构，所以相同数据上的指标可能不同。
此处“最好”限于上述四种结构、指定随机划分和指标排序。
超父的选择和报告使用同一组交叉验证结果，若需估计选择后的最终泛化表现，
应另留独立测试集或采用嵌套交叉验证。
